# Linear Regression
Companion to the linear regression notes. Run the cells in order; each section matches a section of the notes.

## 1. Preparing the lending data
The leakage column `Amount.Funded.By.Investors` is dropped first. Rare categories are grouped into `other` (see the trap box in the notes on why a singular $X^TX$ appears otherwise). 20% of rows are held out as a test set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ld = pd.read_csv("../../data/loan_data_train.csv")
ld = ld.drop(columns=["ID", "Amount.Funded.By.Investors"])   # leakage: decided after application

for col in ["Interest.Rate", "Debt.To.Income.Ratio"]:
    ld[col] = pd.to_numeric(ld[col].str.replace("%", ""), errors="coerce")
for col in ["Amount.Requested", "Open.CREDIT.Lines", "Revolving.CREDIT.Balance"]:
    ld[col] = pd.to_numeric(ld[col], errors="coerce")

ld["Loan.Length"] = pd.to_numeric(ld["Loan.Length"].str.replace(" months", ""), errors="coerce")
fico = ld["FICO.Range"].str.split("-", expand=True).astype(float)
ld["FICO"] = fico.mean(axis=1)                               # midpoint of the range
ld["Employment.Length"] = pd.to_numeric(
    ld["Employment.Length"].str.replace("10+ years", "10", regex=False)
                           .str.replace("< 1 year", "0", regex=False)
                           .str.extract(r"(\d+)")[0], errors="coerce")
ld = ld.drop(columns=["FICO.Range", "State"])
ld = ld.dropna()

for col in ["Home.Ownership", "Loan.Purpose"]:
    counts = ld[col].value_counts()
    rare = counts[counts < 30].index          # too few rows to estimate a weight reliably
    ld[col] = ld[col].replace(dict.fromkeys(rare, "other"))
ld = pd.get_dummies(ld, columns=["Home.Ownership", "Loan.Purpose"], drop_first=True, dtype=float)

y = ld["Interest.Rate"]
X = ld.drop(columns="Interest.Rate")
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
print(x_train.shape, x_test.shape)
x_train.head()

## 2. The gradient and our own SGD
Gradient of the squared error, divided by the batch size, with step decay of the learning rate and a convergence check.

In [ ]:
def gradient(y, x, w):
    return -x.T @ (y - x @ w) / x.shape[0]

def my_lr_sgd(y, x, learning_rate, num_steps, batch=100, decay_rate=0.9, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(x.shape[1])
    prev = np.inf
    for i in range(num_steps):
        idx = rng.choice(x.shape[0], batch)          # a random mini-batch
        w -= learning_rate * gradient(y[idx], x[idx], w)
        if i % 1000 == 0:                            # check convergence now and then
            cost = np.mean((y - x @ w) ** 2)
            if abs(prev - cost) < 1e-6:
                print("converged at iteration", i)
                break
            prev = cost
        if i > 0 and i % 20000 == 0:                 # step decay of the learning rate
            learning_rate *= decay_rate
    return w

## 3. Estimating the weights three ways
Closed form via `np.linalg.solve` (not `inv`), `sklearn` as a benchmark, and SGD on standardised features converted back to the original scale.

In [ ]:
# closed form: solve (X^T X) w = X^T y
x_la = x_train.copy()
x_la.insert(0, "constant", 1.0)
w_la = np.linalg.solve(x_la.T @ x_la, x_la.T @ y_train)

# sklearn puts w0 in intercept_ and the rest in coef_
sk_lr = LinearRegression().fit(x_train, y_train)
w_sk = np.r_[sk_lr.intercept_, sk_lr.coef_]
print("max diff closed form vs sklearn:", np.abs(w_la - w_sk).max())

# SGD on standardised features, then back to the original scale
scaler = StandardScaler().fit(x_train)
x_sd = np.c_[np.ones(len(x_train)), scaler.transform(x_train)]
w_sgd_sd = my_lr_sgd(y_train.values, x_sd, 0.01, 200000)
w_sgd = np.zeros_like(w_sgd_sd)
w_sgd[1:] = w_sgd_sd[1:] / scaler.scale_
w_sgd[0] = w_sgd_sd[0] - np.sum(scaler.mean_ * w_sgd_sd[1:] / scaler.scale_)

print(f"{'':28s} {'closed form':>12s} {'SGD':>10s}")
for name, a, b in zip(x_la.columns, w_la, w_sgd):
    print(f"{name:28s} {a:12.5f} {b:10.5f}")

## 4. Is the model any good?
Everything here is measured on the held-out test rows, against the baseline of predicting the training mean.

In [ ]:
pred = sk_lr.predict(x_test)
print("test  MAE", mean_absolute_error(y_test, pred))
print("test  RMSE", np.sqrt(mean_squared_error(y_test, pred)))
print("test  R2", r2_score(y_test, pred))
print("train MAE", mean_absolute_error(y_train, sk_lr.predict(x_train)))
print("baseline MAE", mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())))

## 5. Looking at the mistakes
A shapeless cloud around zero is what we want. Look for curves, funnels and isolated points.

In [ ]:
import matplotlib.pyplot as plt
resid = y_test - pred
plt.scatter(pred, resid, s=8, alpha=0.5)
plt.axhline(0, color="black", lw=1)
plt.xlabel("predicted interest rate"); plt.ylabel("residual")

## 6. Multicollinearity check
Condition number on raw vs standardised features, and the variance inflation factor (VIF) of each feature.

In [ ]:
xs = np.c_[np.ones(len(x_train)), StandardScaler().fit_transform(x_train)]
print("condition number, raw features:         ", np.linalg.cond(x_la.T @ x_la))
print("condition number, standardised features:", np.linalg.cond(xs.T @ xs))

# VIF_j = 1 / (1 - R^2 of x_j regressed on all other features)
vif = {}
for col in x_train.columns:
    others = x_train.drop(columns=col)
    r2 = LinearRegression().fit(others, x_train[col]).score(others, x_train[col])
    vif[col] = 1 / (1 - r2) if r2 < 1 else np.inf
pd.Series(vif).sort_values(ascending=False)

## Try it yourself
1. Add `State` back as dummies (rare states grouped into `other`). Does test MAE improve? Does train MAE improve more?
2. Run `my_lr_sgd` on the unscaled features with the same learning rate. What happens?
3. Add a squared FICO term and compare the residual plot and test error.
4. Include `Amount.Funded.By.Investors` and compare test MAE. Why is that number meaningless for the business?